# DR1 HEALPix Spectra Exploration

This notebook checks the exact DESI DR1 spectra layout used by `redshifty` and the cleaning cuts that are currently possible from the per-HEALPix `coadd` + `redrock` files.

Main question for this pass: do we already have galaxy/QSO-only, primary, good-redshift rows, or do we need a separate zcatalog-based index?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

HEALPIX_ROOT = Path("/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix")
ZCAT_PATH = Path("/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/zcatalog/v1/zall-pix-iron.fits")

# Match redshifty defaults: surveys=["sv3", "main"], programs=["bright", "dark"].
# Use one concrete main/dark file so exploration is reproducible and fast.
coadd_path = HEALPIX_ROOT / "main" / "dark" / "263" / "26316" / "coadd-main-dark-26316.fits"
redrock_path = coadd_path.with_name(coadd_path.name.replace("coadd-", "redrock-"))

print("coadd:", coadd_path)
print("redrock:", redrock_path)
print("coadd exists:", coadd_path.exists())
print("redrock exists:", redrock_path.exists())
print("zcat exists:", ZCAT_PATH.exists())


In [ ]:
with fits.open(coadd_path, memmap=True) as coadd:
    coadd.info()


In [ ]:
with fits.open(coadd_path, memmap=True) as coadd, fits.open(redrock_path, memmap=True) as redrock:
    fibermap = coadd["FIBERMAP"].data
    redshifts = redrock["REDSHIFTS"].data

    print("Rows")
    print("  FIBERMAP:", len(fibermap))
    print("  REDSHIFTS:", len(redshifts))

    print()
    print("Band array shapes")
    for band in ["B", "R", "Z"]:
        print(f"  {band}_WAVELENGTH:", coadd[f"{band}_WAVELENGTH"].data.shape)
        print(f"  {band}_FLUX:", coadd[f"{band}_FLUX"].data.shape)

    print()
    print("Relevant FIBERMAP columns present")
    for col in ["TARGETID", "OBJTYPE", "COADD_FIBERSTATUS", "DESI_TARGET", "BGS_TARGET", "MWS_TARGET"]:
        print(f"  {col:18s}", col in fibermap.columns.names)

    print()
    print("Relevant REDSHIFTS columns present")
    for col in ["TARGETID", "Z", "ZERR", "ZWARN", "SPECTYPE", "SUBTYPE", "ZCAT_PRIMARY", "OBJTYPE"]:
        print(f"  {col:18s}", col in redshifts.columns.names)


## Current `redshifty` Cleaning

The streaming loader filters each row on the fly using only information available in the per-HEALPix files:

- `ZWARN == 0` from `redrock-*.fits`
- `COADD_FIBERSTATUS == 0` from `coadd-*.fits`
- nonzero summed absolute flux across B/R/Z bands

It does **not** apply `ZCAT_PRIMARY`, because that column is not in the per-HEALPix `REDSHIFTS` table. It also does **not** remove stars unless we add an explicit `SPECTYPE` cut.


In [ ]:
with fits.open(coadd_path, memmap=True) as coadd, fits.open(redrock_path, memmap=True) as redrock:
    fibermap = coadd["FIBERMAP"].data
    redshifts = redrock["REDSHIFTS"].data

    zwarn_good = redshifts["ZWARN"] == 0
    fiber_good = fibermap["COADD_FIBERSTATUS"] == 0
    nonzero_flux = (
        np.abs(coadd["B_FLUX"].data).sum(axis=1)
        + np.abs(coadd["R_FLUX"].data).sum(axis=1)
        + np.abs(coadd["Z_FLUX"].data).sum(axis=1)
    ) > 0
    galaxy_qso = np.isin(redshifts["SPECTYPE"].astype(str), ["GALAXY", "QSO"])
    star = redshifts["SPECTYPE"].astype(str) == "STAR"

    redshifty_good = zwarn_good & fiber_good & nonzero_flux
    our_training_good = redshifty_good & galaxy_qso

    def count(label, mask):
        print(f"{label:35s} {int(mask.sum()):6d} / {len(mask)}")

    count("ZWARN == 0", zwarn_good)
    count("COADD_FIBERSTATUS == 0", fiber_good)
    count("nonzero flux", nonzero_flux)
    count("redshifty current good", redshifty_good)
    count("GALAXY or QSO", galaxy_qso)
    count("STAR", star)
    count("our no-star good", our_training_good)

    print()
    print("SPECTYPE counts after redshifty cuts")
    values, counts = np.unique(redshifts["SPECTYPE"].astype(str)[redshifty_good], return_counts=True)
    for value, n in zip(values, counts):
        print(f"  {value:10s} {int(n):6d}")


## ZCAT Columns

`ZCAT_PRIMARY` lives in the DR1 zcatalog, not in the per-HEALPix redrock file. That means a primary-only, no-stars training set needs either:

1. a precomputed allowed `TARGETID` / row index from `zall-pix-iron.fits`, or
2. a loader that joins each HEALPix row against a zcatalog-derived lookup.

For training speed, precomputing the row index is cleaner.


In [ ]:
with fits.open(ZCAT_PATH, memmap=True) as zcat:
    zcat.info()
    # Use the first table HDU that has columns. This avoids assuming an HDU name.
    table_hdus = [hdu for hdu in zcat if hasattr(hdu, "columns") and hdu.columns is not None]
    cols = table_hdus[0].columns.names
    print()
    print("First table HDU:", table_hdus[0].name)
    print("Column count:", len(cols))
    for col in ["TARGETID", "Z", "ZWARN", "SPECTYPE", "ZCAT_PRIMARY", "OBJTYPE", "SURVEY", "PROGRAM", "HEALPIX"]:
        print(f"  {col:18s}", col in cols)


## Correct Spectrum Indexing

DESI coadd flux arrays in this file are shaped `(n_spectra, n_wavelength)`, so one spectrum is `FLUX[row, :]`, not `FLUX[:, row]`.


In [ ]:
with fits.open(coadd_path, memmap=True) as coadd, fits.open(redrock_path, memmap=True) as redrock:
    fibermap = coadd["FIBERMAP"].data
    redshifts = redrock["REDSHIFTS"].data

    zwarn_good = redshifts["ZWARN"] == 0
    fiber_good = fibermap["COADD_FIBERSTATUS"] == 0
    nonzero_flux = (
        np.abs(coadd["B_FLUX"].data).sum(axis=1)
        + np.abs(coadd["R_FLUX"].data).sum(axis=1)
        + np.abs(coadd["Z_FLUX"].data).sum(axis=1)
    ) > 0
    galaxy_qso = np.isin(redshifts["SPECTYPE"].astype(str), ["GALAXY", "QSO"])
    good = zwarn_good & fiber_good & nonzero_flux & galaxy_qso

    row = int(np.flatnonzero(good)[0])
    targetid = int(redshifts["TARGETID"][row])
    z = float(redshifts["Z"][row])
    spectype = str(redshifts["SPECTYPE"][row])

    plt.figure(figsize=(11, 4))
    for band, color in [("B", "tab:blue"), ("R", "tab:green"), ("Z", "tab:red")]:
        wave = coadd[f"{band}_WAVELENGTH"].data
        flux = coadd[f"{band}_FLUX"].data[row, :]
        mask = coadd[f"{band}_MASK"].data[row, :] != 0
        plt.plot(wave[~mask], flux[~mask], color=color, lw=0.6, label=band)

    plt.title(f"TARGETID={targetid}  SPECTYPE={spectype}  z={z:.4f}")
    plt.xlabel("Wavelength (Angstrom)")
    plt.ylabel("Flux")
    plt.legend(title="Camera")
    plt.tight_layout()
    plt.show()


## Takeaway

For the current foundation-model plan, the per-HEALPix loader is useful but incomplete:

- It already uses DR1 `spectro/redux/iron/healpix` coadds.
- It cleans `ZWARN`, fiber status, and zero-flux rows on the fly.
- It can remove stars with `SPECTYPE in {GALAXY, QSO}` from per-HEALPix `redrock`.
- It cannot apply `ZCAT_PRIMARY` without the DR1 zcatalog.
- For full training, a precomputed clean row index should be faster and safer than repeatedly dropping rows during collation.
